# 🔁 Loop Transformer — Train a Tiny Thinking LLM from Scratch (Free Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nasserml/E-Commerce-dot-net/blob/master/notebooks/Loop_Transformer_Small_LLM_Colab.ipynb)

**What this notebook does:** trains a small (~4M-parameter) decoder-only **large language model from scratch** in PyTorch — no pretrained weights, no Hugging Face model — using a **Loop Transformer** (a.k.a. Universal / weight-tied recurrent Transformer):

- **From scratch:** RMSNorm, RoPE positional encoding, SwiGLU MLP, causal self-attention, a character tokenizer, and the training loop — all implemented right here, no `transformers` library.
- **Loop Transformer:** `n_layers = 4` unique blocks, looped `n_loops = 4` times with **shared weights** → effective depth 16 for the parameter cost of 4. A learned per-loop embedding tells the shared weights which iteration they're on.
- **Thinking:** the model learns to write step-by-step traces inside `<think>...</think>` before answering in `<answer>...</answer>`. Bonus: because weights are loop-tied, you can run **extra loops at inference** (e.g. 8 instead of 4) to give the model more "thinking compute" with zero extra parameters.
- **KV cache:** full key/value caching implemented **per (loop, layer)** so generation stays O(1) per token. We verify cached decoding matches full forward passes exactly, and benchmark the speedup.

**Data:** ~20k synthetic examples generated on the fly — arithmetic with digit-by-digit reasoning traces (75%) + simple English sentences (25%). No downloads required, fully reproducible.

**Requirements:** Colab **free tier** with GPU: `Runtime → Change runtime type → Hardware accelerator → GPU (T4)`.

| Setting | Time on free T4 | Reasoning quality |
|---|---|---|
| `total_steps = 3000` (default) | ~15–25 min | good |
| `total_steps = 8000` | ~45–60 min | better |

> Run cells top to bottom. The only heavy cell is **§8 Train** — everything else runs in seconds.


In [ ]:
# ============================================================
# 0) Environment: GPU check + speed flags + reproducibility
# ============================================================
# If the GPU check below fails, enable one:
# Colab menu: Runtime > Change runtime type > Hardware accelerator > GPU
try:
    import google.colab  # noqa: F401  (only exists on Colab)
    get_ipython().system("nvidia-smi -L")
except Exception:
    pass

import math
import os
import random
import re
import json
import time
from contextlib import nullcontext
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"torch {torch.__version__} | cuda: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    tot = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {tot:.1f} GB | bf16 ok: {torch.cuda.is_bf16_supported()}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
else:
    print("⚠️ No GPU detected — on Colab set Runtime > Change runtime type > T4 GPU.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

try:
    import numpy as np
    print("numpy", np.__version__)
except ImportError:
    raise SystemExit("numpy is required (it is preinstalled on Colab).")

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# ============================================================
# 1) Config — tuned for a FREE Colab T4 (~15 GB VRAM)
#    3000 steps ≈ 15–25 min. For better reasoning try 6000–8000.
# ============================================================
@dataclass
class Config:
    # --- Model: tiny Loop Transformer (implemented from scratch in §5) ---
    d_model: int = 256
    n_layers: int = 4     # UNIQUE layers (this is what you store in VRAM)
    n_heads: int = 4
    n_loops: int = 4      # weight-tied loops -> effective depth = 4*4 = 16
    max_loops: int = 8    # headroom for extra "thinking" loops at inference
    block_size: int = 192 # max sequence length (chars, incl. <think> trace)
    dropout: float = 0.0
    rope_theta: float = 10000.0
    use_checkpoint: bool = False  # True = less VRAM, ~20% slower (checkpoint over loops)

    # --- Training ---
    batch_size: int = 64
    total_steps: int = 3000
    lr: float = 3e-4
    min_lr: float = 3e-5
    warmup_steps: int = 200
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    eval_every: int = 250
    sample_every: int = 750

    # --- Data ---
    n_train: int = 20000
    n_val: int = 1000
    math_frac: float = 0.75  # math-with-thinking vs plain English sentences

    seed: int = 1337


cfg = Config()
for k, v in asdict(cfg).items():
    print(f"{k:16s} {v}")


In [ ]:
# ============================================================
# 2) Tokenizer from scratch (character-level + special tokens)
#    Char-level is ideal here: tiny vocab, great for arithmetic,
#    zero dependencies, trains in milliseconds.
# ============================================================
class CharTokenizer:
    def __init__(self):
        self.specials = ["<pad>", "<unk>", "<bos>", "<eos>",
                         "<think>", "</think>", "<answer>", "</answer>"]
        self.stoi, self.itos = {}, {}
        self._spec_sorted = sorted(self.specials, key=len, reverse=True)

    def train(self, texts):
        chars = sorted(set("".join(texts)))
        chars = [c for c in chars if c not in "<>"]  # <> reserved for specials
        vocab = self.specials + chars
        self.stoi = {t: i for i, t in enumerate(vocab)}
        self.itos = {i: t for t, i in self.stoi.items()}
        print(f"vocab size: {len(vocab)} ({len(self.specials)} specials + {len(chars)} chars)")

    def encode(self, s, add_bos=True, add_eos=True):
        ids = []
        i = 0
        while i < len(s):
            hit = None
            if s[i] == "<":  # all specials start with '<'
                for sp in self._spec_sorted:  # longest match first
                    if s.startswith(sp, i):
                        hit = sp
                        break
            if hit is not None:
                ids.append(self.stoi[hit])
                i += len(hit)
            else:
                ids.append(self.stoi.get(s[i], self.unk_id))
                i += 1
        if add_bos:
            ids = [self.bos_id] + ids
        if add_eos:
            ids = ids + [self.eos_id]
        return ids

    def decode(self, ids):
        return "".join(self.itos.get(i, "<unk>") for i in ids)

    @property
    def pad_id(self): return self.stoi["<pad>"]

    @property
    def unk_id(self): return self.stoi["<unk>"]

    @property
    def bos_id(self): return self.stoi["<bos>"]

    @property
    def eos_id(self): return self.stoi["<eos>"]

    def save(self, path):
        with open(path, "w") as f:
            json.dump({"stoi": self.stoi}, f)

    @classmethod
    def load(cls, path):
        t = cls()
        with open(path) as f:
            t.stoi = json.load(f)["stoi"]
        t.itos = {i: s for s, i in t.stoi.items()}
        return t


# tiny self-test (the real vocab is trained on the dataset in §3)
_t = CharTokenizer()
_t.train(["Q: 1+2=?\n<think>ones: 1+2+0=3.</think>\n<answer>3</answer>"])
_s = "Q: 1+2=?\n<think>ones: 1+2+0=3.</think>\n<answer>3</answer>"
assert _t.decode(_t.encode(_s)) == "<bos>" + _s + "<eos>", "roundtrip failed"
print("tokenizer roundtrip OK | first ids:", _t.encode(_s)[:10])


In [ ]:
# ============================================================
# 3) Data: arithmetic with step-by-step traces + plain sentences
#    Generated locally — no downloads, fully reproducible.
#    Format: Q: 47+58=?\n<think>...steps...</think>\n<answer>105</answer>
# ============================================================
_POS = ["ones", "tens", "hundreds", "thousands"]


def trace_add(a, b):
    sa, sb = str(a), str(b)
    i, j = len(sa) - 1, len(sb) - 1
    carry, steps, digits, p = 0, [], [], 0
    while i >= 0 or j >= 0:
        da = int(sa[i]) if i >= 0 else 0
        db = int(sb[j]) if j >= 0 else 0
        s = da + db + carry
        steps.append(f"{_POS[p]}: {da}+{db}+{carry}={s}, write {s % 10} carry {s // 10}")
        digits.append(str(s % 10))
        carry = s // 10
        i -= 1; j -= 1; p += 1
    if carry:
        steps.append(f"leftover carry {carry}")
        digits.append(str(carry))
    return "; ".join(steps) + f". So {a}+{b}={a + b}."


def trace_sub(a, b):
    assert a >= b
    sa, sb = str(a), str(b)
    i, j = len(sa) - 1, len(sb) - 1
    borrow, steps, digits, p = 0, [], [], 0
    while i >= 0 or j >= 0:
        da = int(sa[i]) if i >= 0 else 0
        db = int(sb[j]) if j >= 0 else 0
        d = da - borrow - db
        if d < 0:
            steps.append(f"{_POS[p]}: {da}-{borrow}-{db} needs borrow, "
                         f"{da + 10}-{borrow}-{db}={d + 10}, write {d + 10} borrow 1")
            d += 10
            borrow = 1
        else:
            steps.append(f"{_POS[p]}: {da}-{borrow}-{db}={d}, write {d} borrow 0")
            borrow = 0
        digits.append(str(d))
        i -= 1; j -= 1; p += 1
    return "; ".join(steps) + f". So {a}-{b}={a - b}."


def trace_mul(a, b):
    if a == 0 or b == 0:
        return f"Anything times 0 is 0. So {a}*{b}=0."
    return f"Add {b} {a} times: {'+'.join([str(b)] * a)}={a * b}. So {a}*{b}={a * b}."


def math_example(rng):
    r = rng.random()
    if r < 0.45:
        a, b = int(rng.integers(0, 100)), int(rng.integers(0, 100))
        q, ans, tr = f"{a}+{b}", str(a + b), trace_add(a, b)
    elif r < 0.75:
        a, b = int(rng.integers(0, 100)), int(rng.integers(0, 100))
        a, b = max(a, b), min(a, b)
        q, ans, tr = f"{a}-{b}", str(a - b), trace_sub(a, b)
    else:
        a, b = int(rng.integers(0, 13)), int(rng.integers(0, 13))
        q, ans, tr = f"{a}*{b}", str(a * b), trace_mul(a, b)
    return f"Q: {q}=?\n<think>{tr}</think>\n<answer>{ans}</answer>", q, ans


_NOUN = ["cat", "dog", "bird", "fish", "frog", "fox", "bear", "bee", "owl", "wolf",
         "tree", "sun", "moon", "star", "lake", "hill", "cloud", "stone"]
_VERB = ["runs", "jumps", "sings", "sleeps", "plays", "eats", "flies", "swims",
         "glows", "dances", "hides", "watches", "shines", "rests"]
_ADJ = ["red", "big", "small", "happy", "quick", "bright", "soft", "loud",
        "calm", "wild", "tiny", "warm", "cold", "green"]
_PREP = ["on", "under", "near", "by", "over", "behind"]


def lang_example(rng):
    n1, n2 = rng.choice(_NOUN, 2, replace=False)
    s1 = (f"The {rng.choice(_ADJ)} {n1} {rng.choice(_VERB)} {rng.choice(_PREP)} "
          f"the {rng.choice(_ADJ)} {n2}.")
    if rng.random() < 0.5:
        s2 = f"It is a {rng.choice(_ADJ)} day and the {n1} feels {rng.choice(_ADJ)}."
        return s1 + " " + s2
    return s1


def build_corpus(n, seed, math_frac):
    rng = np.random.default_rng(seed)
    texts, prompts, answers = [], [], []
    for _ in range(n):
        if rng.random() < math_frac:
            t, q, a = math_example(rng)
            texts.append(t)
            prompts.append(f"Q: {q}=?\n<think>")  # generation prompt (answer held out)
            answers.append(a)
        else:
            texts.append(lang_example(rng))
    return texts, prompts, answers


train_texts, _, _ = build_corpus(cfg.n_train, cfg.seed, cfg.math_frac)
val_texts, val_prompts, val_answers = build_corpus(cfg.n_val, cfg.seed + 999, cfg.math_frac)
print(f"train: {len(train_texts)} texts | val: {len(val_texts)} texts ({len(val_prompts)} math)")

tok = CharTokenizer()
tok.train(train_texts)
print("special ids:", {s: tok.stoi[s] for s in tok.specials})

# length audit: everything must fit block_size (+ bos/eos)
lens = np.array([len(tok.encode(t)) for t in train_texts])
print(f"encoded len: mean {lens.mean():.0f} | max {lens.max()} | "
      f"limit {cfg.block_size + 1} | truncated: {(lens > cfg.block_size + 1).mean() * 100:.2f}%")
bad = sum(1 for t in train_texts if tok.unk_id in tok.encode(t))
print("train examples containing <unk>:", bad, "(expect 0)")

print("\n--- sample math ---")
print(next(t for t in train_texts if t.startswith("Q:")))
print("\n--- sample language ---")
print(next(t for t in train_texts if not t.startswith("Q:")))


In [ ]:
# ============================================================
# 4) PyTorch datasets: pad to block_size, shift labels, mask pads
# ============================================================
class LMData(torch.utils.data.Dataset):
    def __init__(self, texts, tok, block_size):
        self.rows = []
        for t in texts:
            ids = tok.encode(t)[:block_size + 1]
            if len(ids) < 2:
                continue
            x = ids[:-1]
            y = ids[1:]
            pad = block_size - len(x)
            x = x + [tok.pad_id] * pad
            y = y + [-100] * pad  # -100 = ignored by cross_entropy
            self.rows.append((torch.tensor(x), torch.tensor(y)))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


train_ds = LMData(train_texts, tok, cfg.block_size)
val_ds = LMData(val_texts, tok, cfg.block_size)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True,
    num_workers=2, pin_memory=True)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
print(f"batches: {len(train_loader)} train / {len(val_loader)} val")

xb, yb = next(iter(train_loader))
print("x:", tuple(xb.shape), "y:", tuple(yb.shape),
      "| supervised tokens/batch:", int((yb != -100).sum()))
print("decode(x[0]):", tok.decode([i for i in xb[0].tolist() if i != tok.pad_id])[:120], "...")


## How a Loop Transformer works (the 2-minute version)

A standard decoder-only Transformer stacks `L` **different** layers and runs the tokens through once:

```
tokens → [L0] → [L1] → [L2] → [L3] → logits        (depth 4, params of 4 layers)
```

A **Loop Transformer** (Universal Transformer family) reuses the *same* `L` layers `T` times — weights tied across loops — plus a small learned **loop embedding** (an iteration counter) so the shared weights know which pass they're on:

```
loop 0:  x → [L0] → [L1] → [L2] → [L3] ─┐  (+ loop_emb[0])
loop 1:  x → [L0] → [L1] → [L2] → [L3] ─┤  (+ loop_emb[1])
loop 2:  x → [L0] → [L1] → [L2] → [L3] ─┤  (+ loop_emb[2])
loop 3:  x → [L0] → [L1] → [L2] → [L3] ─┴→ logits
                                         (depth 16, params of 4 layers)
```

**Why loop?**
1. **Parameter efficiency** — depth (which drives reasoning) without the memory cost.
2. **Iterative refinement** — each pass can fix mistakes of the previous one, like re-reading your own draft.
3. **Test-time thinking** — train with `T=4`, infer with `T=8`. More compute per token, zero new parameters (§10 measures this).

**Thinking traces.** We train the model to emit `<think>…step-by-step…</think><answer>…</answer>`. The loops do *latent* refinement; the tokens do *explicit* reasoning. Both are visible and measurable here.

**KV cache with loops.** Normal caching stores K/V per layer. With loops, the *same* layer sees *different* representations on each pass, so we store K/V per **(loop, layer)** pair — `T×L` caches. Prefill the prompt once, then decode one token at a time in O(1). §6 proves cached decoding matches full recomputation; §9 benchmarks the speedup.


In [ ]:
# ============================================================
# 5) The Loop Transformer LM — from scratch
#    RMSNorm + RoPE + causal MHA + SwiGLU + loop tying + KV cache
# ============================================================
import torch.utils.checkpoint as _ckpt


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)


def build_rope_cache(head_dim, max_len, theta=10000.0):
    """NeoX-style RoPE frequencies. Returns cos/sin: [max_len, head_dim//2]."""
    assert head_dim % 2 == 0
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    pos = torch.arange(max_len).float()
    freqs = torch.outer(pos, inv)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: [B,H,T,Dh]; cos/sin: [T,Dh/2] already sliced for the right positions."""
    c = cos.to(dtype=x.dtype, device=x.device).unsqueeze(0).unsqueeze(0)  # [1,1,T,Dh/2]
    s = sin.to(dtype=x.dtype, device=x.device).unsqueeze(0).unsqueeze(0)
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = dropout

    def forward(self, x, cos, sin, start_pos=0, past_kv=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = (self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
                   .permute(2, 0, 3, 1, 4).unbind(0))  # each [B,H,T,Dh]
        q = apply_rope(q, cos[start_pos:start_pos + T], sin[start_pos:start_pos + T])
        k = apply_rope(k, cos[start_pos:start_pos + T], sin[start_pos:start_pos + T])
        if past_kv is not None:  # decode: append new K/V to the cache
            pk, pv = past_kv
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)
        present = (k, v) if use_cache else None
        # Prefill (no cache yet): causal over the whole prompt.
        # Decode (1 new token): it may attend to all cached past, no mask needed.
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=(past_kv is None and T > 1),
            dropout_p=self.drop if self.training else 0.0)
        y = y.transpose(1, 2).reshape(B, T, C)
        return self.proj(y), present


class SwiGLU(nn.Module):
    def __init__(self, d_model, mult=4):
        super().__init__()
        hidden = ((mult * d_model + 63) // 64) * 64
        self.gate = nn.Linear(d_model, hidden, bias=False)
        self.up = nn.Linear(d_model, hidden, bias=False)
        self.down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class LoopBlock(nn.Module):
    """One pre-norm Transformer block. Each layer instance is unique,
    but the SAME instances are reused on every loop (weight tying)."""
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model)

    def forward(self, x, cos, sin, start_pos=0, past_kv=None, use_cache=False):
        h, present = self.attn(self.ln1(x), cos, sin, start_pos, past_kv, use_cache)
        x = x + h
        x = x + self.mlp(self.ln2(x))
        return x, present


class LoopTransformerLM(nn.Module):
    """Tiny Loop (Universal / weight-tied) Transformer LM.

    - `n_layers` unique blocks; the whole stack is repeated `n_loops`
      times with shared weights + a learned per-loop embedding.
    - KV cache is kept per (loop, layer): cache[t][li] = (K, V).
    - At inference you may pass a larger `n_loops` ("thinking harder").
    """
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.cfg = cfg
        self.n_layers, self.n_heads = cfg.n_layers, cfg.n_heads
        self.n_loops, self.max_loops = cfg.n_loops, cfg.max_loops
        self.block_size = cfg.block_size

        self.tok_emb = nn.Embedding(vocab_size, cfg.d_model)
        self.loop_emb = nn.Embedding(cfg.max_loops, cfg.d_model)  # learned loop counter
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList(
            [LoopBlock(cfg.d_model, cfg.n_heads, cfg.dropout) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # tied input/output embeddings

        cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.block_size, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, past_kv=None, use_cache=False, n_loops=None, start_pos=0):
        n_loops = self.n_loops if n_loops is None else n_loops
        if not 1 <= n_loops <= self.max_loops:
            raise ValueError(f"n_loops must be in [1, {self.max_loops}], got {n_loops}")
        B, T = input_ids.shape
        if start_pos + T > self.block_size:
            raise ValueError(f"length {start_pos + T} exceeds block_size {self.block_size}")
        x = self.drop(self.tok_emb(input_ids))
        cos, sin = self.rope_cos, self.rope_sin
        ckpt = self.cfg.use_checkpoint and self.training and not use_cache and past_kv is None
        new_cache = [] if use_cache else None

        for t in range(n_loops):
            x = x + self.loop_emb.weight[t].view(1, 1, -1)  # "this is loop t"
            loop_cache = [] if use_cache else None
            for li, layer in enumerate(self.layers):
                past = past_kv[t][li] if past_kv is not None else None
                if ckpt:
                    def _f(_x, _layer=layer, _cos=cos, _sin=sin, _sp=start_pos):
                        out, _ = _layer(_x, _cos, _sin, _sp, None, False)
                        return out
                    x = _ckpt.checkpoint(_f, x, use_reentrant=False)
                else:
                    x, present = layer(x, cos, sin, start_pos, past, use_cache)
                    if use_cache:
                        loop_cache.append(present)
            if use_cache:
                new_cache.append(loop_cache)
        return self.head(self.norm(x)), new_cache

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=64, temperature=0.7, top_k=50, top_p=0.0,
                 use_cache=True, n_loops=None, eos_id=None):
        """Autoregressive decoding. temperature<=0 = greedy. Set eos_id to stop early."""
        self.eval()
        n_loops = self.n_loops if n_loops is None else n_loops
        logits, past = self(input_ids, past_kv=None, use_cache=use_cache,
                            n_loops=n_loops, start_pos=0)
        out = input_ids
        neg_inf = torch.tensor(float("-inf"), device=input_ids.device)
        for _ in range(max_new_tokens):
            last = logits[:, -1, :]
            if temperature is None or temperature <= 0:  # greedy
                nxt = last.argmax(-1, keepdim=True)
            else:
                l = last / temperature
                if top_k and top_k < l.size(-1):
                    thr = torch.topk(l, top_k).values[:, [-1]]
                    l = torch.where(l < thr, neg_inf, l)
                if top_p and 0 < top_p < 1.0:
                    s, idx = torch.sort(l, descending=True)
                    probs = F.softmax(s, dim=-1)
                    mask = (probs.cumsum(-1) - probs) > top_p
                    s = torch.where(mask, neg_inf, s)
                    l = torch.full_like(l, float("-inf")).scatter(1, idx, s)
                nxt = torch.multinomial(F.softmax(l, dim=-1), 1)
            out = torch.cat([out, nxt], dim=1)
            if out.size(1) >= self.block_size:
                break
            if eos_id is not None and bool((nxt == eos_id).all()):
                break
            if use_cache:
                logits, past = self(nxt, past_kv=past, use_cache=True,
                                    n_loops=n_loops, start_pos=out.size(1) - 1)
            else:
                logits, _ = self(out, use_cache=False, n_loops=n_loops, start_pos=0)
        return out


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def kv_cache_mb(n_loops, batch=1, seq=None, dtype_bytes=2):
    seq = seq or cfg.block_size
    n = 2 * batch * n_loops * cfg.n_layers * cfg.d_model * seq * dtype_bytes
    return n / 1e6  # K and V per (loop, layer): d_model elems per token


model = LoopTransformerLM(cfg, vocab_size=len(tok.stoi)).to(device)
print(f"params: {count_params(model) / 1e6:.2f}M | "
      f"{cfg.n_layers} layers x {cfg.n_loops} loops = effective depth {cfg.n_layers * cfg.n_loops}")
print(f"KV cache fp16, batch=1, seq={cfg.block_size}: "
      f"{kv_cache_mb(cfg.n_loops):.2f} MB @train loops, {kv_cache_mb(cfg.max_loops):.2f} MB @max loops")


In [ ]:
# ============================================================
# 6) Sanity checks: shapes, loss, KV-cache equivalence, loop range
# ============================================================
model.eval()
with torch.no_grad():
    xb0, yb0 = next(iter(val_loader))
    xb0, yb0 = xb0[:4].to(device), yb0[:4].to(device)
    logits0, _ = model(xb0)
    print("logits shape:", tuple(logits0.shape), "(batch, seq, vocab)")
    l0 = F.cross_entropy(logits0.reshape(-1, logits0.size(-1)), yb0.reshape(-1), ignore_index=-100)
    print(f"untrained loss: {float(l0):.3f}  (random ≈ ln(V) = {math.log(len(tok.stoi)):.3f})")

    # --- KV-cache equivalence: incremental decode == one full forward ---
    prompt = "Q: 12+29=?\n<think>ones: "
    ids = torch.tensor([tok.encode(prompt, add_eos=False)], device=device)
    full, _ = model(ids)  # [1, T, V]
    k = ids.size(1) // 2
    pre_lg, past = model(ids[:, :k], use_cache=True, start_pos=0)
    parts = [pre_lg[:, -1:, :]]  # predicts token k
    for j in range(k, ids.size(1)):  # decode rest token-by-token
        lg, past = model(ids[:, j:j + 1], past_kv=past, use_cache=True, start_pos=j)
        parts.append(lg)  # predicts token j+1
    inc = torch.cat(parts, dim=1)
    ref = full[:, k - 1:, :]  # logits predicting tokens k..T-1 (+eos slot)
    diff = (inc.float() - ref.float()).abs().max().item()
    match = (inc.argmax(-1) == ref.argmax(-1)).float().mean().item()
    print(f"cache-vs-full max|Δlogit| = {diff:.2e} | argmax match = {match * 100:.0f}%")
    assert diff < 1e-2, "KV cache diverges from full forward!"

    # --- variable loop counts (test-time thinking range) ---
    for L in (1, cfg.n_loops, cfg.max_loops):
        lg, cache = model(xb0[:1], use_cache=True, n_loops=L)
        assert len(cache) == L and all(len(lc) == cfg.n_layers for lc in cache)
        print(f"n_loops={L}: ok, cache holds {L}x{cfg.n_layers} (loop x layer) KV pairs")

    # --- untrained sample (expect gibberish — that's the "before" picture) ---
    ids = torch.tensor([tok.encode("Q: 7+5=?\n<think>", add_eos=False)], device=device)
    out = model.generate(ids, max_new_tokens=60, temperature=0.0)
    print("untrained sample:", repr(tok.decode(out[0].tolist()))[:220])


In [ ]:
# ============================================================
# 7) Optimizer, cosine schedule, AMP + eval helpers
# ============================================================
def lr_schedule(step):
    if step < cfg.warmup_steps:
        return cfg.lr * step / max(1, cfg.warmup_steps)
    p = (step - cfg.warmup_steps) / max(1, cfg.total_steps - cfg.warmup_steps)
    p = min(1.0, max(0.0, p))
    return cfg.min_lr + 0.5 * (cfg.lr - cfg.min_lr) * (1 + math.cos(math.pi * p))


optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                              weight_decay=cfg.weight_decay, betas=(0.9, 0.95))
use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16  # T4 has no bf16 -> fp16
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda" and not use_bf16))
print(f"AMP dtype: {amp_dtype} | GradScaler: {scaler.is_enabled()}")


def extract_answer(text):
    m = re.search(r"<answer>(.*?)</answer>", text, re.S)
    return m.group(1).strip() if m else None


@torch.no_grad()
def eval_loss(n_batches=20):
    model.eval()
    tot, n = 0.0, 0
    it = iter(val_loader)
    for _ in range(n_batches):
        try:
            xb, yb = next(it)
        except StopIteration:
            break
        xb, yb = xb.to(device), yb.to(device)
        logits, _ = model(xb)
        tot += F.cross_entropy(logits.reshape(-1, logits.size(-1)), yb.reshape(-1),
                               ignore_index=-100).item()
        n += 1
    return tot / max(1, n)


@torch.no_grad()
def eval_accuracy(n=48, n_loops=None, max_new_tokens=150, verbose=0):
    """Greedy think-then-answer accuracy on held-out math questions."""
    model.eval()
    n = min(n, len(val_prompts))
    correct = 0
    for i in range(n):
        ids = torch.tensor([tok.encode(val_prompts[i], add_eos=False)], device=device)
        out = model.generate(ids, max_new_tokens=max_new_tokens, temperature=0.0,
                             use_cache=True, n_loops=n_loops, eos_id=tok.eos_id)
        pred = extract_answer(tok.decode(out[0].tolist()))
        ok = (pred == val_answers[i])
        correct += ok
        if verbose and (i < verbose or not ok):
            print(f"{'✓' if ok else '✗'} {val_prompts[i]!r} → {pred!r} (true {val_answers[i]!r})")
    return correct / max(1, n)


@torch.no_grad()
def sample_show(prompts, max_new_tokens=150, temperature=0.0, n_loops=None, top_k=50):
    model.eval()
    for p in prompts:
        ids = torch.tensor([tok.encode(p, add_eos=False)], device=device)
        out = model.generate(ids, max_new_tokens=max_new_tokens, temperature=temperature,
                             top_k=top_k, use_cache=True, n_loops=n_loops, eos_id=tok.eos_id)
        print("—" * 70)
        print(tok.decode(out[0].tolist()))
    print("—" * 70)


In [ ]:
# ============================================================
# 8) Train (~15–25 min on a free T4 for 3000 steps)
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

loader = iter(train_loader)
hist = {"step": [], "loss": [], "estep": [], "val": [], "acc": []}
best_val, t0 = float("inf"), time.time()
run_loss, run_n = 0.0, 0

model.train()
for step in range(1, cfg.total_steps + 1):
    try:
        xb, yb = next(loader)
    except StopIteration:
        loader = iter(train_loader)
        xb, yb = next(loader)
    xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)

    for pg in optimizer.param_groups:
        pg["lr"] = lr_schedule(step)
    optimizer.zero_grad(set_to_none=True)
    ctx = torch.amp.autocast("cuda", dtype=amp_dtype) if device == "cuda" else nullcontext()
    with ctx:
        logits, _ = model(xb)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), yb.reshape(-1), ignore_index=-100)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
    scaler.step(optimizer)
    scaler.update()

    run_loss += loss.item()
    run_n += 1
    if step % 50 == 0 or step == 1:
        dt = (time.time() - t0) / 60
        print(f"step {step:5d}/{cfg.total_steps}  train {run_loss / run_n:.4f}  "
              f"lr {lr_schedule(step):.1e}  [{dt:.1f} min]", flush=True)
        hist["step"].append(step)
        hist["loss"].append(run_loss / run_n)
        run_loss, run_n = 0.0, 0

    if step % cfg.eval_every == 0 or step == cfg.total_steps:
        v = eval_loss()
        acc = eval_accuracy(n=48)
        hist["estep"].append(step)
        hist["val"].append(v)
        hist["acc"].append(acc)
        print(f"  ↳ val loss {v:.4f} | think-answer acc {acc * 100:5.1f}% "
              f"(ppl {math.exp(min(v, 20)):.1f})", flush=True)
        if v < best_val and step < cfg.total_steps:
            best_val = v
            torch.save({"step": step, "model": model.state_dict(),
                        "opt": optimizer.state_dict(), "cfg": asdict(cfg),
                        "stoi": tok.stoi}, "best_loop_llm.pt")
            print("  ↳ ✅ saved best_loop_llm.pt", flush=True)
        model.train()

    if step % cfg.sample_every == 0 or step == cfg.total_steps:
        sample_show(["Q: 47+58=?\n<think>", "Q: 9*7=?\n<think>", "The happy cat"],
                    max_new_tokens=120)

print(f"\nDone in {(time.time() - t0) / 60:.1f} min. Final think-answer acc: {hist['acc'][-1] * 100:.1f}%")

# --- curves ---
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist["step"], hist["loss"], label="train")
ax[0].set(xlabel="step", ylabel="loss", title="Training loss")
ax[0].legend()
ax[0].grid(alpha=.3)
ax[1].plot(hist["estep"], hist["val"], label="val loss")
ax[1].legend(loc="upper left")
ax[1].grid(alpha=.3)
ax2 = ax[1].twinx()
ax2.plot(hist["estep"], hist["acc"], color="tab:orange", marker="o", label="val acc")
ax2.set(ylabel="accuracy", title="Validation")
ax2.legend(loc="upper right")
fig.tight_layout()
fig.savefig("curves.png", dpi=120)
try:
    from IPython.display import Image, display
    display(Image("curves.png"))
except Exception:
    print("saved curves.png")


In [ ]:
# ============================================================
# 9) KV-cache: prove it matches full recompute + measure speedup
# ============================================================
model.eval()
prompt = "Q: 83+47=?\n<think>"
ids = torch.tensor([tok.encode(prompt, add_eos=False)], device=device)

with torch.no_grad():
    t_cached = tok.decode(model.generate(
        ids, max_new_tokens=130, temperature=0.0, use_cache=True)[0].tolist())
    t_full = tok.decode(model.generate(
        ids, max_new_tokens=130, temperature=0.0, use_cache=False)[0].tolist())
print("identical outputs:", t_cached == t_full)
print(t_cached)


# --- benchmark: greedy, 100 new tokens ---
def bench(use_cache, iters=3, new_tokens=100):
    if device == "cuda":
        torch.cuda.synchronize()
    ts = []
    with torch.no_grad():
        for _ in range(iters):
            s = time.time()
            model.generate(ids, max_new_tokens=new_tokens, temperature=0.0, use_cache=use_cache)
            if device == "cuda":
                torch.cuda.synchronize()
            ts.append(time.time() - s)
    return sum(ts) / len(ts)


with torch.no_grad():
    c = bench(True)
    f = bench(False)
print(f"\ncached   : {c:.2f}s for 100 tokens ({100 / c:.0f} tok/s)")
print(f"no cache : {f:.2f}s for 100 tokens ({100 / f:.0f} tok/s)")
print(f"speedup  : {f / c:.1f}x  (grows with sequence length)")


In [ ]:
# ============================================================
# 10) Thinking at test time: more loops = more compute, same params
#     (model was trained with n_loops=4; max_loops=8 allows scaling)
# ============================================================
print("Sweeping inference loops:")
for L in [1, 2, 4, 6, 8]:
    acc = eval_accuracy(n=100, n_loops=L)
    print(f"  n_loops={L}: think-answer acc = {acc * 100:5.1f}%", flush=True)

print("\nSame question, loops=1 vs loops=8 (greedy):")
sample_show(["Q: 68+27=?\n<think>"], n_loops=1)
sample_show(["Q: 68+27=?\n<think>"], n_loops=8)

print("\nNo-think baseline (direct answer — a format never seen in training):")
sample_show(["Q: 68+27=?\n<answer>"], n_loops=4, max_new_tokens=12)


In [ ]:
# ============================================================
# 11) Save everything, download, (optional) push to Hugging Face
# ============================================================
torch.save({"step": cfg.total_steps, "model": model.state_dict(),
            "opt": optimizer.state_dict(), "cfg": asdict(cfg),
            "stoi": tok.stoi}, "loop_llm_final.pt")
tok.save("loop_tok.json")
print("saved loop_llm_final.pt + loop_tok.json")

# Download to your laptop (Colab only)
try:
    from google.colab import files
    files.download("loop_llm_final.pt")
    files.download("loop_tok.json")
    print("downloads started")
except Exception as e:
    print("not on Colab (or download blocked):", e)

# --- Reload later (any machine with torch + the class definitions above) ---
print("""
# ckpt = torch.load("loop_llm_final.pt", map_location="cpu", weights_only=False)
# cfg  = Config(**ckpt["cfg"])
# tok  = CharTokenizer(); tok.stoi = ckpt["stoi"]; tok.itos = {i: s for s, i in tok.stoi.items()}
# model = LoopTransformerLM(cfg, len(tok.stoi)); model.load_state_dict(ckpt["model"]); model.eval()
""")

# --- Optional: push to Hugging Face Hub ---
# !pip install -q huggingface_hub
# from huggingface_hub import HfApi
# HfApi().create_repo("loop-transformer-tiny", exist_ok=True)
# HfApi().upload_file("loop_llm_final.pt", path_in_repo="loop_llm_final.pt",
#                     repo_id="<your-username>/loop-transformer-tiny")


## Next steps & experiments

You now have a from-scratch Loop Transformer that **thinks in tokens**, **refines in loops**, and **decodes with a KV cache**. Ideas to push it further:

1. **Train longer** — set `total_steps = 8000` and watch the loops=8 advantage grow; log at which step explicit traces become reliably correct.
2. **Bigger loop gap** — train with `n_loops=2`, infer with `n_loops=8`: how much test-time compute can compensate for less training compute?
3. **Real text** — swap the synthetic sentences for TinyStories (`!pip install datasets`, stream 50k stories) and mix with the math traces for a more general tiny LM.
4. **BPE tokenizer** — replace `CharTokenizer` with a 2k-token BPE (`!pip install tokenizers`) and compare sample quality vs. training speed.
5. **Adaptive thinking** — add per-token halting (ACT): a small head predicting "loop again / stop", so easy tokens exit after 1–2 loops and hard ones use all 8.
6. **GQA + sliding window** — convert the attention to grouped-query attention to shrink the KV cache, then raise `block_size` to 512.
7. **Distill it** — train a `n_layers=2, n_loops=2` student to imitate this model's traces (sequence-level KD) and measure what survives.

If you publish results, the interesting plots are: **accuracy vs. inference loops** (§10) and **tok/s vs. sequence length with/without cache** (§9).
